# Big coding · Multi-head self-attention from scratch

**Goal:** implement causal multi-head self-attention, then verify outputs and gradients against PyTorch. **Time:** 60–90 minutes. **Prerequisites:** tensor shapes, softmax, autograd. **Runtime:** CPU, Python 3 + PyTorch; no model downloads.

Save a copy in Drive before editing. Run cells in order. Starter tests intentionally fail until you complete the TODO. A separate sample solution is folded in section 4; use **Show code** to expand it.

## 1. Problem and tensor contract

Implement `self_attention(x, wq, wk, wv, wo, heads)` without built-in attention functions.

- Input `x`: `[B, T, D]`, with positive `T` and `D % heads == 0`.
- Each weight is `[D, D]`, applied as `x @ weight` (no bias).
- Split Q/K/V to `[B, heads, T, Dh]`, where `Dh = D // heads`.
- Compute scaled dot products, mask future keys, normalize over keys, combine values, merge heads, and apply `wo`.
- Return output `[B, T, D]` and attention probabilities `[B, heads, T, T]`.
- Keep the autograd graph intact. Assume no padding and no dropout. This is a single layer, without positions, residuals, or normalization.

**Deliverables:** passing output and gradient tests, an explanation of each axis, and a short account of the quadratic attention matrix cost.


In [ ]:
import math
import torch
import torch.nn.functional as F

torch.manual_seed(11)
torch.set_num_threads(1)

def oracle(x, wq, wk, wv, wo, heads):
    b, t, d = x.shape
    q, k, v = [(x @ w).reshape(b, t, heads, d // heads).transpose(1, 2)
               for w in (wq, wk, wv)]
    out = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=True)
    return out.transpose(1, 2).contiguous().reshape(b, t, d) @ wo


## 2. Your implementation

Write out the operations rather than calling the oracle. Use a mask whose True entries mean **forbidden future positions** in your `masked_fill`. Different PyTorch attention APIs use different Boolean-mask conventions.

Hints: transpose the last two axes of K; scale by `sqrt(Dh)`, not `sqrt(D)`; use `contiguous()` or `reshape()` after transposing heads back.


In [ ]:
def self_attention(x, wq, wk, wv, wo, heads):
    # TODO: projections -> heads -> scaled scores -> causal mask -> softmax -> output.
    raise NotImplementedError('Implement self_attention, then run the next cell.')


## 3. Correctness tests

Check multiple batch/head/length shapes, causal visibility, row normalization, future-token isolation, first-token behavior, and gradients with respect to the input and all four weights. Float64 reduces numerical noise in this small exercise.


In [ ]:
def check_attention(fn):
    for b, t, d, h in [(2, 7, 16, 4), (1, 1, 8, 2), (3, 5, 12, 1)]:
        torch.manual_seed(19)
        x = torch.randn(b, t, d, dtype=torch.float64, requires_grad=True)
        weights = [(torch.randn(d, d, dtype=torch.float64) / math.sqrt(d)).requires_grad_() for _ in range(4)]
        out, probs = fn(x, *weights, h)
        expected = oracle(x, *weights, h)
        assert out.shape == (b, t, d) and probs.shape == (b, h, t, t)
        assert torch.isfinite(out).all() and (probs >= 0).all()
        torch.testing.assert_close(out, expected, atol=1e-9, rtol=1e-7)
        torch.testing.assert_close(probs.sum(-1), torch.ones(b, h, t, dtype=x.dtype))
        assert torch.count_nonzero(probs.triu(1)) == 0
        torch.testing.assert_close(out[:, :1], (x[:, :1] @ weights[2]) @ weights[3])
        target = torch.randn_like(out)
        actual_grads = torch.autograd.grad((out * target).sum(), (x, *weights))
        expected_grads = torch.autograd.grad((expected * target).sum(), (x, *weights))
        for actual, reference in zip(actual_grads, expected_grads):
            torch.testing.assert_close(actual, reference, atol=1e-8, rtol=1e-6)
        if t > 1:
            changed = x.detach().clone()
            changed[:, t//2:] += 5
            altered, _ = fn(changed, *weights, h)
            torch.testing.assert_close(out[:, :t//2], altered[:, :t//2], atol=1e-9, rtol=1e-7)
    print('PASS: shapes, causal masking, normalization, outputs, gradients, future isolation.')

check_attention(self_attention)


## 4. Sample solution — expand when ready

Use **Show code** on the folded cell. Each operation is annotated; the solution remains separate from your implementation.


In [ ]:
#@title Sample solution — expand to review { display-mode: "form" }
def reference_self_attention(x, wq, wk, wv, wo, heads):
    b, t, d = x.shape  # Read batch size, sequence length, and model width.
    dh = d // heads  # Each head receives an equal slice of the model width.
    # Project all tokens; reshape introduces the head axis, then transpose puts it second.
    q, k, v = [(x @ w).reshape(b, t, heads, dh).transpose(1, 2) for w in (wq, wk, wv)]
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(dh)  # [B, H, query, key].
    future = torch.ones(t, t, dtype=torch.bool, device=x.device).triu(1)  # Exclude j > i.
    scores = scores.masked_fill(future, -torch.inf)  # Broadcasting applies this mask to every batch/head.
    probs = scores.softmax(dim=-1)  # Every query distributes probability across visible keys.
    context = probs @ v  # Weighted sums over values give [B, H, T, Dh].
    merged = context.transpose(1, 2).contiguous().reshape(b, t, d)  # Rejoin the heads per token.
    output = merged @ wo  # Mix information across heads with the final learned projection.
    return output, probs  # Preserve both output and probabilities for the tests.

check_attention(reference_self_attention)  # Reuse the exact student test harness.


## 5. Inspect attention and memory

Inspect one query row and calculate score-tensor storage. This counts just one dense score matrix, not all activation memory, gradients, or optimized fused-kernel memory.


In [ ]:
x_demo = torch.randn(1, 6, 16)
w_demo = [torch.randn(16, 16) / 4 for _ in range(4)]
_, attention = reference_self_attention(x_demo, *w_demo, 4)
print('Head 0, query 3:', attention[0, 0, 3].detach())
for length in (128, 256, 512):
    print(f'T={length}: one float32 [B=2,H=4,T,T] matrix uses {2*4*length*length*4:,} bytes')


## 6. Review and extensions

1. Why is the softmax axis the key axis? Why divide by the square root of the head width?
2. Why must the diagonal remain visible? What happens when a query has no valid keys?
3. Compare causal and bidirectional attention. Add a padding mask without creating fully masked rows.
4. Add a module with trainable projection layers, then connect it to the KV cache exercise.

Source: [PyTorch scaled-dot-product attention](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html). This oracle uses `is_causal=True`, avoiding Boolean-mask convention ambiguity.
